In [1]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter.BayesianLinearRegression import BayesianLinearRegression
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample

from pyro.infer import Predictive
import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pyro.set_rng_seed(1)

%matplotlib inline
plt.style.use('default')


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)

X = df[["cont_africa","rugged","cont_africa_x_rugged"]]
y = df[["rgdppc_2000"]]

x_data, y_data = data[:, :-1], data[:, -1]


In [3]:
linear_model = BayesianLinearRegression(
    len(X.columns), # parent's number
    num_iterations=500, 
    lr=0.03,
    posterior_samples=100,
) 


In [4]:
x_data, y_data = data[:, :-1], data[:, -1]


In [5]:
linear_model.fit(X, y)


[iteration 0001] loss: 4.4480
[iteration 0101] loss: 3.0572
[iteration 0201] loss: 2.5214
[iteration 0301] loss: 1.9759
[iteration 0401] loss: 1.6874


BayesianLinearRegression(in_features=3, num_iterations=500,
                         posterior_samples=100)

In [21]:
median = linear_model.guide.median(x_data, y_data)

for name, value in median.items():
    print(name, ": ", value.detach().cpu().numpy())

# sigma :  1.0184335
# linear.weight :  [[-1.7521899  -0.12638536  0.22601697]]
# linear.bias :  [8.998347]

# regression function: y∼Normal(8.998347−1.7521899x1−0.12638536x2+0.22601697x1x2,1.0184335)


sigma :  1.0184335
linear.weight :  [[-1.7521899  -0.12638536  0.22601697]]
linear.bias :  [8.998347]


In [22]:
import pandas as pd

X_df = pd.DataFrame(
    x_data[:5].detach().cpu().numpy(),
    columns=["x1", "x2", "x3"],
)

pred_dist = linear_model.predict_proba(X_df)


In [23]:
pred_dist


Normal(columns=Index(['y'], dtype='object'),
       index=RangeIndex(start=0, stop=5, step=1),
       mu=array([[7.419653],
       [8.52572 ],
       [8.936822],
       [8.987578],
       [8.487305]], dtype=float32),
       sigma=array([[0.87969804],
       [0.99002546],
       [0.97307414],
       [0.90368116],
       [1.037179  ]], dtype=float32))

In [24]:
pred_dist.mu.shape


(5, 1)